### Motivation behind multi attention heads 

A single self-attention computes one attention head which captures one kind of relationship. However, with multi attention heads, with each having its own Q, K, V projections and concatenated ouputs and each head operating in a smaller subsapce of dimension d_model / n_heads, which improves the expressive power.

Multi-head attention is the default for every transformer, but difference lies with how many heads and whether keys and values share projections (Grouped-Query Attention, Multi-Query Attention, Multi-head Latent Attention)

##### Variations in Attention Types 

* Variant 1: Multi-head Attention (MHA): Requires N K/V Heads
-- Used in GPT2/ BERT/ T5
* Variant 2: Multi-Query Attention (MQA): Requires 1 K/V Head (Re-use the same K/V across all heads)
-- Used in PaLM/ Falcon
* Variant 3: Grouped-Query Attention (GQA): Requires G K/V Heads (Reuses the K/V G times)
-- Used in LLama/ Qwen/ Mistral

* Variant 4: Multi-head latent (MLA) ~ Distinct concept from the other 3 Attention styles 
-- Used in Deepseek

In [1]:
import math
import random 
from typing import List 

In [34]:
class Matrix:
    __slots__ = ('rows', 'cols', 'data')

    def __init__(self, rows, cols, fill = 0.8, data = None):
        self.rows = rows
        self.cols = cols
        if data is not None:
            self.data = data 
        else:
            self.data = [fill] * (self.rows * self.cols)
    
    def get(self, i, j):
        return self.data[i * self.cols + j]
    
    def set(self, i, j, v):
        self.data[i * self.cols + j] = v
    
    def row(self, i):
        return self.data[i * self.cols : (i + 1) * self.cols]


In [ ]:
def randn_matrix(rows, cols, rng, scale=None):
    if scale is None: 
        scale = math.sqrt(2.0 / (rows + cols))
    m = Matrix(rows, cols)
    for i in range(rows * cols):
        m.data[i] = rng.gauss(0.0, scale)
    return m

In [36]:
### Optimised version for multiplication

def matmul(A, B):
    assert A.cols == B.rows, f"{A.cols} != {B.rows}"
    out = Matrix(A.rows, B.cols)
    for i in range(A.rows):
        for k in range(A.cols):
            aik = A.get(i, k)
            if aik == 0.0:
                continue
            base_i = i * B.cols
            base_k = k * B.cols
            for j in range(B.cols):
                out.data[base_i + j] += aik * B.data[base_k + j]

    return out

## Textbook version for better understanding 
def matmul(A, B):
    assert A.cols == B.rows
    out = Matrix(A.rows, B.cols)
    for i in range(A.rows):          # row of A
        for j in range(B.cols):      # column of B
            s = 0.0
            for k in range(A.cols):  # summation dimension
                s += A.get(i, k) * B.get(k, j)
            out.set(i, j, s)

    return out

In [37]:
def transpose(A):
    out = Matrix(A.cols, A.rows)
    for i in range(A.rows):
        for j in range(A.cols):
            out.set(j, i, A.get(i,j))

    return out

In [38]:
def softmax_rows(A):
    out = Matrix(A.rows, A.cols)
    for i in range(A.rows):
        row = A.row(i)
        m = max(row)
        exps = [math.exp(x - m) for x in row]
        s = sum(exps)
        for j, e in enumerate(exps):
            out.set(i, j, e/s)
    
    return out

In [45]:
def scaled_dot_product_attention(Q, K, V):
    dk = Q.cols
    scale = 1.0 / math.sqrt(dk)
    scores = matmul(Q, transpose(K))
    for i in range(scores.rows * scores.cols):
        scores.data[i] *= scale
    weights = softmax_rows(scores)
    out = matmul(weights, V)
    return out, weights 

In [46]:
### Optimised version of multi-head attention - single matrix multiplication but subsequently broken down
###  into individuals heads

def split_heads(X, n_heads):
    assert X.cols % n_heads == 0, "d_model not divisible by n_heads"
    d_head = X.cols // n_heads
    heads = []
    for h in range(n_heads):
        H = Matrix(X.rows, d_head)
        for i in range(X.rows):
            for j in range(d_head):
                H.set(i, j, X.get(i, h * d_head + j))
        heads.append(H)
    
    return heads

def combine_heads(heads):
    n = heads[0].rows
    d_head = heads[0].cols
    d_model = d_head * len(heads)
    out = Matrix(n, d_model)
    for h, H in enumerate(heads):
        for i in range(n):
            for j in range(d_head):
                out.set(i, h * d_head + j, H.get(i,j))
    
    return out

In [47]:
def multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads):
    Q = matmul(X, Wq)
    K = matmul(X, Wk)
    V = matmul(X, Wv)
    Qh = split_heads(Q, n_heads)
    Kh = split_heads(K, n_heads)
    Vh = split_heads(V, n_heads)
    
    head_outs = []
    per_head_weights = []
    for q, k, v in zip(Qh, Kh, Vh):
        o, w = scaled_dot_product_attention(q, k, v)
        head_outs.append(o)
        per_head_weights.append(w)
    concat = combine_heads(head_outs)
    return matmul(concat, Wo), per_head_weights

In [48]:
def grouped_query_attention(X, Wq, Wk, Wv, Wo, n_heads, n_kv_heads):
    Q = matmul(X, Wq)
    K = matmul(X, Wk)
    V = matmul(X, Wv)
    Qh = split_heads(Q, n_heads)
    Kh_small = split_heads(K, n_kv_heads)
    Vh_small = split_heads(V, n_kv_heads)
    repeat = n_heads // n_kv_heads
    Kh = [Kh_small[i // repeat] for i in range(n_heads)]
    Vh = [Vh_small[i // repeat] for i in range(n_heads)]

    head_outs = []
    for q, k, v in zip(Qh, Kh, Vh):
        o, w = scaled_dot_product_attention(q, k, v)
        head_outs.append(o)
    concat = combine_heads(head_outs)
    return matmul(concat, Wo)

In [49]:
def print_matrix(name, M: Matrix, width=6, prec=3):
    print(f"-- {name} ({M.rows}x{M.cols}) --")
    for i in range(M.rows):
        row = M.row(i)
        print("  " + "  ".join(f"{v:>{width}.{prec}f}" for v in row))


def main():
    rng = random.Random(42)
    tokens = ["the", "cat", "sat", "on", "the", "mat"]
    n = len(tokens)
    d_model = 8
    n_heads = 2

    X = randn_matrix(n, d_model, rng, scale=1.0)
    Wq = randn_matrix(d_model, d_model, rng)
    Wk = randn_matrix(d_model, d_model, rng)
    Wv = randn_matrix(d_model, d_model, rng)
    Wo = randn_matrix(d_model, d_model, rng)

    out, weights = multi_head_attention(X, Wq, Wk, Wv, Wo, n_heads=n_heads)

    print(f"=== multi-head attention: {n_heads} heads, d_model={d_model}, d_head={d_model // n_heads} ===")
    print(f"input  shape: ({X.rows}, {X.cols})")
    print(f"output shape: ({out.rows}, {out.cols})")
    print()
    for h, W in enumerate(weights):
        print(f"-- head {h} attention weights --")
        print(f"{'':>6}", end="")
        for t in tokens:
            print(f"{t:>7}", end="")
        print()
        for i in range(n):
            print(f"{tokens[i]:>6}", end="")
            for j in range(n):
                print(f"{W.get(i, j):>7.3f}", end="")
            print()
        print()

    # GQA demo: 4 Q heads, 2 KV heads
    d_model = 8
    n_heads = 4
    n_kv = 2
    Wq = randn_matrix(d_model, d_model, rng)
    Wk = randn_matrix(d_model, (d_model // n_heads) * n_kv, rng)
    Wv = randn_matrix(d_model, (d_model // n_heads) * n_kv, rng)
    Wo = randn_matrix(d_model, d_model, rng)
    out_gqa = grouped_query_attention(X, Wq, Wk, Wv, Wo, n_heads=n_heads, n_kv_heads=n_kv)
    print(f"=== GQA: {n_heads} Q heads, {n_kv} KV heads ===")
    print(f"output shape: ({out_gqa.rows}, {out_gqa.cols})")
    kv_cache_full = n_heads * n * (d_model // n_heads) * 2
    kv_cache_gqa = n_kv * n * (d_model // n_heads) * 2
    print(f"KV cache elements (MHA):  {kv_cache_full}")
    print(f"KV cache elements (GQA):  {kv_cache_gqa}  ({kv_cache_full // kv_cache_gqa}x smaller)")


In [50]:
main()

=== multi-head attention: 2 heads, d_model=8, d_head=4 ===
input  shape: (6, 8)
output shape: (6, 8)

-- head 0 attention weights --
          the    cat    sat     on    the    mat
   the  0.147  0.214  0.262  0.154  0.121  0.102
   cat  0.113  0.230  0.217  0.156  0.219  0.065
   sat  0.125  0.158  0.092  0.132  0.352  0.142
    on  0.103  0.245  0.346  0.154  0.103  0.048
   the  0.132  0.238  0.389  0.112  0.053  0.077
   mat  0.177  0.177  0.228  0.161  0.111  0.146

-- head 1 attention weights --
          the    cat    sat     on    the    mat
   the  0.180  0.161  0.136  0.217  0.167  0.139
   cat  0.141  0.167  0.109  0.210  0.257  0.116
   sat  0.186  0.171  0.141  0.166  0.116  0.219
    on  0.140  0.179  0.143  0.297  0.119  0.121
   the  0.153  0.095  0.153  0.222  0.339  0.039
   mat  0.144  0.171  0.209  0.157  0.157  0.163

=== GQA: 4 Q heads, 2 KV heads ===
output shape: (6, 8)
KV cache elements (MHA):  96
KV cache elements (GQA):  48  (2x smaller)


In [53]:
import torch.nn as nn 

mha = nn.MultiheadAttention(embed_dim = 512, num_heads = 8, batch_first = True)

In [62]:
mha.__dict__['_parameters']['in_proj_weight']

Parameter containing:
tensor([[ 0.0328,  0.0155,  0.0393,  ...,  0.0166, -0.0396,  0.0325],
        [ 0.0237,  0.0532,  0.0152,  ...,  0.0497, -0.0366, -0.0248],
        [ 0.0082,  0.0291, -0.0004,  ..., -0.0307, -0.0392, -0.0469],
        ...,
        [-0.0082, -0.0182, -0.0311,  ..., -0.0312, -0.0446,  0.0464],
        [-0.0015,  0.0260, -0.0463,  ..., -0.0324, -0.0136,  0.0396],
        [ 0.0257,  0.0157, -0.0323,  ...,  0.0269, -0.0067,  0.0240]],
       requires_grad=True)